In [2]:
# Cell 1: Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("✅ Libraries loaded.")

✅ Libraries loaded.


In [3]:
# Cell 2: Load training and live datasets
train = pd.read_csv("../Dataset/high_salary.csv")
live = pd.read_csv("../Dataset/high_salary.live.csv")

print("Train shape:", train.shape)
print("Live shape:", live.shape)
display(train.head())

Train shape: (20900, 19)
Live shape: (6967, 18)


,id,social-security-number,house-number,age-group,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capitalgain,capitalloss,hoursperweek,native-country-code,native-country,label
0,8616,552701574.0,9854.0,2.0,self-emp-inc,270079.0,bachelors,13.0,married-civ-spouse,exec-managerial,husband,white,male,0.0,0.0,3.0,USA,united-states,1.0
1,21982,956556990.0,7588.0,3.0,local-gov,146325.0,doctorate,16.0,married-civ-spouse,prof-specialty,husband,white,male,0.0,2.0,2.0,USA,united-states,1.0
2,11191,958358623.0,6729.0,0.0,private,240767.0,hs-grad,9.0,never-married,other-service,not-in-family,white,female,0.0,0.0,1.0,USA,united-states,0.0
3,22229,224206693.0,6288.0,2.0,private,118536.0,hs-grad,9.0,divorced,machine-op-inspct,other-relative,black,male,0.0,0.0,2.0,USA,united-states,0.0
4,20732,276413230.0,8276.0,3.0,private,160440.0,bachelors,13.0,married-civ-spouse,sales,husband,white,male,0.0,0.0,3.0,USA,united-states,1.0


In [4]:
# Cell 3: Split features and target
X = train.drop(columns=['label'])
y = train['label']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['id', 'social-security-number', 'house-number', 'age-group', 'fnlwgt', 'education-num', 'capitalgain', 'capitalloss', 'hoursperweek']
Categorical columns: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country-code', 'native-country']


In [5]:
# Cell 4: Preprocessing for numeric + categorical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

In [6]:
# Cell 5: Split for validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])

Train samples: 16720
Validation samples: 4180


In [10]:
# Cell 6: Baseline Random Forest (before tuning)
rf_base = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

rf_base.fit(X_train, y_train)
y_pred = rf_base.predict(X_val)

print("Baseline Accuracy:", accuracy_score(y_val, y_pred))
print("Baseline F1:", f1_score(y_val, y_pred))

Baseline Accuracy: 0.815311004784689
Baseline F1: 0.7830241708825182


In [8]:
# Cell 7: Grid Search to tune Random Forest
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}

rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)
print("✅ Grid search completed.")
print("Best Parameters:", grid_search.best_params_)
print("Best CV F1 Score:", grid_search.best_score_)


✅ Grid search completed.
Best Parameters: {'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}
Best CV F1 Score: 0.7920295236934362


In [9]:
# Cell 8: Evaluate tuned Random Forest on validation data
best_rf = grid_search.best_estimator_

y_val_pred = best_rf.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation F1:", f1_score(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))

Validation Accuracy: 0.8246411483253588
Validation F1: 0.7981272376755716

Classification Report:
               precision    recall  f1-score   support

         0.0       0.87      0.82      0.84      2427
         1.0       0.77      0.83      0.80      1753

    accuracy                           0.82      4180
   macro avg       0.82      0.82      0.82      4180
weighted avg       0.83      0.82      0.83      4180



In [ ]:
# Cell 9: Retrain best model on all data and predict live dataset
best_rf.fit(X, y)

# Ensure id column exists
if 'id' not in live.columns:
    live = live.reset_index().rename(columns={'index': 'id'})

live_features = live[X.columns]
preds = best_rf.predict(live_features)

output = pd.DataFrame({
    'id': live['id'],
    'prediction': preds
})

GROUP_NAME = "G19"  # <-- Change this
filename = f"{GROUP_NAME}_predictions.live.csv"
output.to_csv(filename, index=False)

print(f"✅ Predictions saved as {filename}")
display(output.head())

In [ ]:
# Cell 10: Save tuned Random Forest model
import joblib
joblib.dump(best_rf, f"{GROUP_NAME}_best_rf.joblib")
print(f"💾 Model saved as {GROUP_NAME}_best_rf.joblib")